<a href="https://colab.research.google.com/github/iamaanahmad/RuView/blob/add-colab-notebook/colab_ruview_ngrok_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === Cell 0: ngrok auth token ===
# To get your ngrok auth token:
# 1. Sign up/log in at https://ngrok.com/
# 2. Go to 'Your Authtoken' in the dashboard: https://dashboard.ngrok.com/get-started/your-authtoken
# 3. Copy your authtoken and paste it below.
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN_HERE":
    print("⚠️ Paste your ngrok token into NGROK_AUTH_TOKEN, then re-run this cell.")
else:
    print("✅ ngrok token set (showing prefix only):", NGROK_AUTH_TOKEN[:10] + "...")

In [ ]:
# === Cell 1: install deps ===
!pip -q install pyngrok requests

print("✅ Installed pyngrok + requests")

In [ ]:
# === Cell 2: clone repo ===
import os, subprocess, textwrap, sys

REPO_URL = "https://github.com/ruvnet/RuView"
REPO_DIR = "RuView"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print("✅ Repo already cloned")

print("📁 Repo dir:", os.path.abspath(REPO_DIR))

In [ ]:
# === Cell 3: find likely server entrypoints ===
import os, re, pathlib, json

root = pathlib.Path("RuView")

candidates = []

# Common patterns
patterns = [
    ("Cargo.toml", r"\[package\]"),
    ("docker", r"Dockerfile"),
    ("python-fastapi", r"FastAPI\("),
    ("uvicorn", r"uvicorn"),
    ("axum", r"axum"),
    ("rocket", r"rocket::"),
]

# Scan a subset of files (avoid huge scan)
for p in root.rglob("*"):
    if p.is_dir():
        continue
    if p.suffix.lower() not in [".md", ".toml", ".rs", ".py", ".yml", ".yaml", ".json", ".sh", ".ts", ".js"]:
        continue
    # Skip big files
    try:
        if p.stat().st_size > 2_000_000:
            continue
        txt = p.read_text(errors="ignore")
    except Exception:
        continue

    hit = False
    for tag, rx in patterns:
        if re.search(rx, txt):
            hit = True
            break
    if hit:
        candidates.append(str(p))

# Print a small curated list (top 60)
print("Found candidate files (showing up to 60):")
for f in candidates[:60]:
    print(" -", f)

print("\nNext: we will choose the correct server command based on what we find.")

In [ ]:
# === Cell 4: run cargo from the Rust workspace root (v2/) ===

import os, shutil, subprocess

# Ensure current process sees cargo in PATH (even if shell wasn't restarted)
if shutil.which("cargo") is None:
    # rustup wrote this file; sourcing it would normally happen in a shell
    cargo_env = os.path.expanduser("~/.cargo/env")
    if os.path.exists(cargo_env):
        # Minimal PATH fix for this Python process
        os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ.get("PATH", "")
print("cargo:", shutil.which("cargo"))

# Move into the Rust workspace directory
%cd /content/RuView/v2

# Confirm Cargo.toml exists here
!ls -la | head -n 50
!test -f Cargo.toml && echo "✅ Found v2/Cargo.toml" || (echo "❌ Cargo.toml still missing" && exit 1)

print("\n--- Checking sensing-server help ---")
!cargo run -q -p wifi-densepose-sensing-server -- --help | head -n 120

In [ ]:
# === Cell 5: hardcode SENSING_ALLOWED_HOSTS for ngrok ===

import subprocess, time, socket, threading, shlex, os

HTTP_PORT = 3000
BIND_ADDR = "0.0.0.0"
UI_PATH = "../ui"

# Put your ngrok hostname here (NO https://, just host).
# After running Cell 6 with your NGROK_AUTH_TOKEN, you will see a public URL.
# For example, if the public URL is 'https://rewire-confirm-humongous.ngrok-free.dev',
# then your NGROK_HOST should be 'rewire-confirm-humongous.ngrok-free.dev'.
NGROK_HOST = "YOUR_NGROK_HOST_HERE"

def is_port_open(port, host="127.0.0.1"):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0
    finally:
        s.close()

# Hardcode all common Host header variants that may reach the server via ngrok/proxies
SENSING_ALLOWED_HOSTS = ",".join([
    NGROK_HOST,
    f"{NGROK_HOST}:443",
    f"{NGROK_HOST}:80",
    f"{NGROK_HOST}:{HTTP_PORT}",
    "localhost",
    f"localhost:{HTTP_PORT}",
    "127.0.0.1",
    f"127.0.0.1:{HTTP_PORT}",
])

env = os.environ.copy()
env["SENSING_ALLOWED_HOSTS"] = SENSING_ALLOWED_HOSTS

cmd = [
    "cargo", "run", "-q", "-p", "wifi-densepose-sensing-server", "--",
    "--bind-addr", BIND_ADDR,
    "--http-port", str(HTTP_PORT),
    "--ui-path", UI_PATH,
    "--source", "simulate",
]

print("Starting sensing-server:\n ", " ".join(shlex.quote(x) for x in cmd))
print("SENSING_ALLOWED_HOSTS =", env["SENSING_ALLOWED_HOSTS"])

server_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

# Stream logs
log_lines = []
def pump_logs():
    for line in server_proc.stdout:
        line = line.rstrip()
        log_lines.append(line)
        print(line)

t = threading.Thread(target=pump_logs, daemon=True)
t.start()

print(f"\nWaiting for HTTP listener on 127.0.0.1:{HTTP_PORT} ...")
for i in range(60):
    if is_port_open(HTTP_PORT, "127.0.0.1"):
        print(f"\n✅ Server is listening at http://127.0.0.1:{HTTP_PORT}")
        break
    if i % 10 == 0:
        print(f"  ... {i}/60 seconds")
    time.sleep(1)

In [ ]:
# === Cell 6: ngrok tunnel ===
from pyngrok import ngrok

if NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN_HERE":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(3000, "http")
    print("✅ Public URL:", public_url)
else:
    print("⚠️ No token set; skipping ngrok")

In [ ]:
# === Cell 7: test the correct endpoints for wifi-densepose-sensing-server ===
import requests, json

base = "http://127.0.0.1:3000"

paths = [
    "/health",
    "/ui/index.html",

    # Pose + zones
    "/api/v1/pose/current",
    "/api/v1/pose/stats",
    "/api/v1/pose/zones/summary",

    # Vital signs
    "/api/v1/vital-signs",
    "/api/v1/edge-vitals",

    # Stream + introspection
    "/api/v1/stream/status",
    "/api/v1/introspection/snapshot",

    # Model info (may be empty if no model loaded)
    "/api/v1/model/info",
    "/api/v1/models",
    "/api/v1/models/active",
]

for path in paths:
    url = base + path
    try:
        r = requests.get(url, timeout=8)
        print(f"{path} -> {r.status_code}")

        ctype = r.headers.get("content-type", "")
        if "application/json" in ctype:
            print(" ", json.dumps(r.json(), indent=2)[:1200])
        else:
            # show a small snippet for HTML/text
            print(" ", r.text[:200].replace("\n", " ") + ("..." if len(r.text) > 200 else ""))
    except Exception as e:
        print(f"{path} -> ERROR: {type(e).__name__}: {str(e)[:200]}")
    print()